In [4]:
import pandas as pd
import matplotlib.pyplot as plt


# df = pd.read_csv('/Users/nrcase/CSC522/CSC522-Project/dataset_with_labels.csv')
df = pd.read_csv('../../datasets/CA_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",1,0,2,CA,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.000000,0.2480,0.576,138.008,4,About_Average
1,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,2,0,3,CA,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.000000,0.1410,0.214,101.061,4,Higher
2,3GCdLUSnKSMJhs4Tj6CV3s,All The Stars (with SZA),"Kendrick Lamar, SZA",3,1,19,CA,2025-02-17,90,True,...,-4.946,1,0.0599,0.0612,0.000195,0.0926,0.557,96.782,4,About_Average
3,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",4,2,-3,CA,2025-02-17,98,False,...,-7.777,0,0.0304,0.3080,0.000000,0.1220,0.535,157.969,3,Lower
4,0aB0v4027ukVziUGwVGYpG,tv off (feat. lefty gunplay),"Kendrick Lamar, Lefty Gunplay",5,0,5,CA,2025-02-17,92,True,...,-6.679,0,0.2630,0.0837,0.000000,0.4230,0.548,100.036,4,Higher


In [5]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [6]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
import xgboost as xgb

preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    xgb.XGBRegressor(tree_method='hist', max_bin=255)
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("XGBoost")
mean_squared_error(y_test, y_pred)

XGBoost


62.51258087158203

In [7]:
from sklearn.model_selection import GridSearchCV
# set up our search grid
param_grid = {"xgbregressor__max_depth":    [3, 4, 5, 6, 7, 8],
              "xgbregressor__n_estimators": [100, 500],
              "xgbregressor__learning_rate": [0.01, 0.015, 0.02, 0.025, 0.03],
              "xgbregressor__min_child_weight": [1,3,5,10],}

# try out every combination of the above values
search = GridSearchCV(pipeline, param_grid, cv=5, scoring="neg_mean_squared_error").fit(X_train, y_train)

print("The best hyperparameters are ",search.best_params_)

The best hyperparameters are  {'xgbregressor__learning_rate': 0.03, 'xgbregressor__max_depth': 8, 'xgbregressor__min_child_weight': 5, 'xgbregressor__n_estimators': 500}


In [11]:
search.best_score_

np.float64(-59.7815658569336)

In [9]:
best_pipeline = make_pipeline(
    preprocessing,
    xgb.XGBRegressor(max_depth=search.best_params_["xgbregressor__max_depth"],
                     n_estimators=search.best_params_["xgbregressor__n_estimators"],
                     learning_rate=search.best_params_["xgbregressor__learning_rate"],
                     min_child_weight=search.best_params_["xgbregressor__min_child_weight"],
                     tree_method='hist',
                     max_bin=255,)
)
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)
print("XGBoost with best hyperparameters")
print(mean_squared_error(y_test, y_pred))


XGBoost with best hyperparameters
60.018497467041016
